# 📊 Análisis Exploratorio - Portfolio de Propiedades

Este notebook analiza la estructura del archivo Excel del módulo Portfolio.

## Objetivo
Procesar, limpiar, depurar, transformar y analizar los datos del portfolio de propiedades para generar informes ejecutivos en formato tabular.


## 1. Importación de librerías


In [83]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuración de pandas para mejor visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✅ Librerías importadas correctamente")


✅ Librerías importadas correctamente


## 2. Carga del archivo Excel


In [84]:
# Ruta al archivo Excel
excel_path = '../data/portfolio/adinco_export_20251201122116.xlsx'  # Ajustar nombre del archivo según corresponda

# Cargar el archivo Excel
try:
    # Primero, verificar qué hojas tiene el Excel
    excel_file = pd.ExcelFile(excel_path)
    print(f"📁 Archivo: {excel_path}")
    print(f"📑 Hojas disponibles: {excel_file.sheet_names}")
    print(f"\nTotal de hojas: {len(excel_file.sheet_names)}")
    
    # Cargar todas las hojas en un diccionario
    dataframes = {}
    for hoja in excel_file.sheet_names:
        df_temp = pd.read_excel(excel_path, sheet_name=hoja)
        dataframes[hoja] = df_temp
        print(f"✅ {hoja}: {df_temp.shape[0]} filas × {df_temp.shape[1]} columnas")
    
except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo en {excel_path}")
    print("Por favor, coloca el archivo Excel del portfolio en la carpeta data/portfolio/")
    dataframes = {}
except Exception as e:
    print(f"❌ Error al cargar el archivo: {e}")
    dataframes = {}


📁 Archivo: ../data/portfolio/adinco_export_20251201122116.xlsx
📑 Hojas disponibles: ['adinco_export_20251201122116']

Total de hojas: 1
✅ adinco_export_20251201122116: 200 filas × 24 columnas


## 3. Limpieza de Datos


In [85]:
# Columnas a eliminar
columnas_a_eliminar = [
    'Id Aviso',
    'Superficie cubierta',
    'Fondo libre',
    'Antiguedad',
    'Cocheras descubiertas',
    'Cocheras cubiertas',
    'Cocheras semicubiertas',
    'Plantas',
    'Nombre de propietario',
    'Email de propietario',
    'Celular de propietario'
]

# Eliminar columnas de todas las hojas
if len(dataframes) > 0:
    resumen_limpieza = []
    
    for nombre_hoja, df in dataframes.items():
        filas_antes = len(df)
        columnas_antes = len(df.columns)
        
        # Eliminar filas completamente vacías
        df = df.dropna(how='all')
        
        # Eliminar columnas especificadas (solo si existen)
        columnas_existentes = [col for col in columnas_a_eliminar if col in df.columns]
        columnas_no_existentes = [col for col in columnas_a_eliminar if col not in df.columns]
        
        if columnas_existentes:
            df = df.drop(columns=columnas_existentes)
            print(f"✅ {nombre_hoja}: Eliminadas {len(columnas_existentes)} columnas")
            if columnas_no_existentes:
                print(f"   ⚠️ Columnas no encontradas: {', '.join(columnas_no_existentes)}")
        else:
            print(f"⚠️ {nombre_hoja}: No se encontraron columnas para eliminar")
        
        # Actualizar el dataframe en el diccionario
        dataframes[nombre_hoja] = df
        
        filas_despues = len(df)
        columnas_despues = len(df.columns)
        filas_eliminadas = filas_antes - filas_despues
        
        resumen_limpieza.append({
            'Hoja': nombre_hoja,
            'Filas Antes': filas_antes,
            'Filas Después': filas_despues,
            'Filas Eliminadas': filas_eliminadas,
            'Columnas Antes': columnas_antes,
            'Columnas Después': columnas_despues,
            'Columnas Eliminadas': columnas_antes - columnas_despues
        })
    
    df_resumen_limpieza = pd.DataFrame(resumen_limpieza)
    print("\n📊 RESUMEN DE LIMPIEZA")
    print("=" * 80)
    display(df_resumen_limpieza)
else:
    print("⚠️ No hay datos cargados para limpiar")


✅ adinco_export_20251201122116: Eliminadas 11 columnas

📊 RESUMEN DE LIMPIEZA


,Hoja,Filas Antes,Filas Después,Filas Eliminadas,Columnas Antes,Columnas Después,Columnas Eliminadas
0,adinco_export_20251201122116,200,200,0,24,13,11


## 4.5. Procesamiento de Superficie Total


In [86]:
def extraer_superficie_numerica(superficie):
    """
    Extrae el valor numérico de la columna Superficie total.
    Maneja formatos como "100 m2", "100 m²", "100", etc.
    Ignora todo desde la "m" en adelante (incluyendo m2, m², etc.)
    Retorna: valor numérico (float) o None
    """
    if pd.isna(superficie):
        return None
    
    # Convertir a string para procesar
    superficie_str = str(superficie).strip()
    
    # Buscar la posición de "m" o "M" (ignorar todo desde ahí)
    pos_m = -1
    for i, char in enumerate(superficie_str):
        if char.lower() == 'm':
            pos_m = i
            break
    
    # Si hay una "m", tomar solo lo que está antes
    if pos_m > 0:
        superficie_str = superficie_str[:pos_m].strip()
    
    # Extraer solo los números, puntos y comas
    numeros = re.sub(r'[^\d.,]', '', superficie_str)
    
    # Formato latino: punto (.) es separador de miles, coma (,) es separador decimal
    # Si hay coma, es el separador decimal
    if ',' in numeros:
        # Eliminar todos los puntos (separadores de miles)
        numeros = numeros.replace('.', '')
        # Reemplazar coma por punto para el formato float de Python
        numeros = numeros.replace(',', '.')
    else:
        # Si no hay coma, los puntos son separadores de miles, eliminarlos
        numeros = numeros.replace('.', '')
    
    try:
        superficie_numerica = float(numeros) if numeros else None
        return superficie_numerica
    except:
        return None

# Procesar la columna Superficie total en todas las hojas
if len(dataframes) > 0:
    resumen_superficie = []
    
    for nombre_hoja, df in dataframes.items():
        if 'Superficie total' in df.columns:
            print(f"\n📊 Procesando columna 'Superficie total' en {nombre_hoja}")
            print("=" * 80)
            
            # Aplicar la función a cada fila
            df['Superficie_total_num'] = df['Superficie total'].apply(extraer_superficie_numerica)
            
            # Resumen de procesamiento
            total_registros = len(df)
            registros_con_superficie = df['Superficie_total_num'].notna().sum()
            registros_sin_superficie = total_registros - registros_con_superficie
            
            print(f"\nResumen de procesamiento:")
            print(f"  Total registros: {total_registros}")
            print(f"  Con superficie válida: {registros_con_superficie} ({registros_con_superficie/total_registros*100:.1f}%)")
            print(f"  Sin superficie válida: {registros_sin_superficie} ({registros_sin_superficie/total_registros*100:.1f}%)")
            
            # Mostrar algunos ejemplos
            print(f"\nEjemplos de procesamiento:")
            ejemplos_superficie = df[['Superficie total', 'Superficie_total_num']].head(10)
            display(ejemplos_superficie)
            
            # Estadísticas de superficie
            if registros_con_superficie > 0:
                print(f"\nEstadísticas de superficie (m²):")
                print(df['Superficie_total_num'].describe())
            
            # Actualizar el dataframe
            dataframes[nombre_hoja] = df
            
            resumen_superficie.append({
                'Hoja': nombre_hoja,
                'Total registros': total_registros,
                'Con superficie válida': registros_con_superficie,
                'Sin superficie válida': registros_sin_superficie
            })
        else:
            print(f"⚠️ {nombre_hoja}: No se encontró la columna 'Superficie total'")
    
    if resumen_superficie:
        df_resumen_superficie = pd.DataFrame(resumen_superficie)
        print("\n📊 RESUMEN DE PROCESAMIENTO DE SUPERFICIE")
        print("=" * 80)
        display(df_resumen_superficie)
else:
    print("⚠️ No hay datos cargados para procesar")



📊 Procesando columna 'Superficie total' en adinco_export_20251201122116

Resumen de procesamiento:
  Total registros: 200
  Con superficie válida: 100 (50.0%)
  Sin superficie válida: 100 (50.0%)

Ejemplos de procesamiento:


,Superficie total,Superficie_total_num
0,-,NaN
1,171 m2,171.0
2,275 m2,275.0
3,-,NaN
4,248 m2,248.0
5,1.050 m2,1050.0
6,462 m2,462.0
7,-,NaN
8,290 m2,290.0
9,-,NaN



Estadísticas de superficie (m²):
count      100.00000
mean       925.30000
std       1904.76167
min          7.00000
25%        167.00000
50%        362.50000
75%       1000.00000
max      15000.00000
Name: Superficie_total_num, dtype: float64

📊 RESUMEN DE PROCESAMIENTO DE SUPERFICIE


,Hoja,Total registros,Con superficie válida,Sin superficie válida
0,adinco_export_20251201122116,200,100,100


## 4. Procesamiento de Precio y Tipo de Operación


In [87]:
import re

def procesar_precio_y_tipo(precio):
    """
    Procesa la columna Precio para detectar:
    - Si contiene "U$D" o "USD" → Tipo: "Venta" (en dólares)
    - Si contiene "consultar precio" (cualquier formato) → Tipo: "Venta"
    - Si el precio es $0 o 0 → Tipo: "Venta"
    - Si es solo número → Tipo: "Alquiler" (en pesos)
    Retorna: (precio_numerico, tipo_operacion)
    """
    if pd.isna(precio):
        return None, None
    
    # Convertir a string para procesar
    precio_str = str(precio).strip()
    precio_str_upper = precio_str.upper()
    
    # CASO 1: Detectar "consultar precio" (case-insensitive)
    if 'CONSULTAR PRECIO' in precio_str_upper:
        return None, 'Venta'
    
    # CASO 2: Detectar si contiene "U$D" o "USD" (puede estar en diferentes formatos)
    if 'U$D' in precio_str_upper or 'USD' in precio_str_upper:
        # Es una venta en dólares
        # Extraer solo los números, puntos y comas
        numeros = re.sub(r'[^\d.,]', '', precio_str)
        # Reemplazar comas por puntos para decimales (formato argentino: 350.000)
        numeros = numeros.replace('.', '').replace(',', '.')
        try:
            precio_numerico = float(numeros) if numeros else None
            return precio_numerico, 'Venta'
        except:
            return None, 'Venta'
    
    # CASO 3: Detectar $0 o 0 (precio cero)
    # Limpiar el string de cualquier carácter no numérico excepto punto y coma
    numeros_temp = re.sub(r'[^\d.,]', '', precio_str)
    numeros_temp = numeros_temp.replace('.', '').replace(',', '.')
    try:
        precio_temp = float(numeros_temp) if numeros_temp else None
        if precio_temp == 0:
            return 0.0, 'Venta'
    except:
        pass
    
    # CASO 4: Es un alquiler en pesos (solo número)
    # Limpiar el string de cualquier carácter no numérico excepto punto y coma
    numeros = re.sub(r'[^\d.,]', '', precio_str)
    # Reemplazar comas por puntos para decimales
    numeros = numeros.replace(',', '.')
    try:
        precio_numerico = float(numeros) if numeros else None
        return precio_numerico, 'Alquiler'
    except:
        return None, 'Alquiler'

# Procesar la columna Precio en todas las hojas
if len(dataframes) > 0:
    resumen_precio = []
    
    for nombre_hoja, df in dataframes.items():
        if 'Precio' in df.columns:
            print(f"\n📊 Procesando columna 'Precio' en {nombre_hoja}")
            print("=" * 80)
            
            # Aplicar la función a cada fila
            resultados = df['Precio'].apply(procesar_precio_y_tipo)
            df['Precio_numerico'] = resultados.apply(lambda x: x[0])
            df['Tipo de operación'] = resultados.apply(lambda x: x[1])
            
            # Resumen de tipos de operación
            resumen_tipos = df['Tipo de operación'].value_counts()
            print(f"\nDistribución de tipos de operación:")
            for tipo, cantidad in resumen_tipos.items():
                print(f"  {tipo}: {cantidad}")
            
            # Mostrar algunos ejemplos
            print(f"\nEjemplos de procesamiento:")
            ejemplos = df[['Precio', 'Precio_numerico', 'Tipo de operación']].head(10)
            display(ejemplos)
            
            # Estadísticas de precios
            print(f"\nEstadísticas de precios:")
            print(f"  Alquileres (pesos): {df[df['Tipo de operación'] == 'Alquiler']['Precio_numerico'].describe()}")
            print(f"  Ventas (dólares): {df[df['Tipo de operación'] == 'Venta']['Precio_numerico'].describe()}")
            
            # Actualizar el dataframe
            dataframes[nombre_hoja] = df
            
            resumen_precio.append({
                'Hoja': nombre_hoja,
                'Total registros': len(df),
                'Alquileres': len(df[df['Tipo de operación'] == 'Alquiler']),
                'Ventas': len(df[df['Tipo de operación'] == 'Venta']),
                'Sin precio': len(df[df['Precio_numerico'].isna()])
            })
        else:
            print(f"⚠️ {nombre_hoja}: No se encontró la columna 'Precio'")
    
    if resumen_precio:
        df_resumen_precio = pd.DataFrame(resumen_precio)
        print("\n📊 RESUMEN DE PROCESAMIENTO DE PRECIOS")
        print("=" * 80)
        display(df_resumen_precio)
else:
    print("⚠️ No hay datos cargados para procesar")



📊 Procesando columna 'Precio' en adinco_export_20251201122116

Distribución de tipos de operación:
  Venta: 144
  Alquiler: 56

Ejemplos de procesamiento:


,Precio,Precio_numerico,Tipo de operación
0,650000,650000.0,Alquiler
1,U$D 350.000,350000.0,Venta
2,U$D 200.000,200000.0,Venta
3,U$D 130.000,130000.0,Venta
4,U$D 50.000,50000.0,Venta
5,U$D 40.000,40000.0,Venta
6,U$D 60.000,60000.0,Venta
7,320000,320000.0,Alquiler
8,U$D 40.000,40000.0,Venta
9,U$D 150.000,150000.0,Venta



Estadísticas de precios:
  Alquileres (pesos): count    5.600000e+01
mean     6.769643e+05
std      4.167741e+05
min      3.000000e+05
25%      4.000000e+05
50%      5.725000e+05
75%      7.575000e+05
max      2.500000e+06
Name: Precio_numerico, dtype: float64
  Ventas (dólares): count       112.000000
mean     101538.839286
std      122874.304733
min           0.000000
25%       14750.000000
50%       65000.000000
75%      135000.000000
max      650000.000000
Name: Precio_numerico, dtype: float64

📊 RESUMEN DE PROCESAMIENTO DE PRECIOS


,Hoja,Total registros,Alquileres,Ventas,Sin precio
0,adinco_export_20251201122116,200,56,144,32


## 5. Análisis y KPIs del Portfolio


## 6. Geocodificación y Visualización en Mapa


In [ ]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time
import folium
from folium.plugins import MarkerCluster

# Inicializar geocodificador (gratis, sin API key)
geolocator = Nominatim(user_agent="portfolio_inmobiliario")

def geocodificar_direccion(direccion, max_intentos=3):
    """
    Geocodifica una dirección usando Nominatim (OpenStreetMap).
    Retorna: (latitud, longitud) o (None, None) si falla
    """
    if pd.isna(direccion) or direccion == '':
        return None, None
    
    for intento in range(max_intentos):
        try:
            location = geolocator.geocode(direccion, timeout=10)
            if location:
                return location.latitude, location.longitude
            else:
                return None, None
        except (GeocoderTimedOut, GeocoderServiceError) as e:
            if intento < max_intentos - 1:
                time.sleep(2)  # Esperar más tiempo en caso de error
                continue
            else:
                return None, None
        except Exception as e:
            return None, None
    
    return None, None

# Procesar geocodificación en todas las hojas
if len(dataframes) > 0:
    for nombre_hoja, df in dataframes.items():
        if 'Dirección' in df.columns and 'Ubicación' in df.columns:
            print(f"\n📊 Geocodificando direcciones en {nombre_hoja}")
            print("=" * 80)
            
            # Crear dirección completa
            df['direccion_completa'] = df['Dirección'].astype(str) + ", " + df['Ubicación'].astype(str)
            
            # Inicializar columnas de coordenadas
            df['latitud'] = None
            df['longitud'] = None
            
            total = len(df)
            geocodificadas = 0
            fallidas = 0
            
            print(f"Total de propiedades a geocodificar: {total}")
            print("⚠️ Esto puede tomar varios minutos (1 segundo por propiedad)...")
            print("Procesando...\n")
            
            # Geocodificar cada propiedad
            for idx, row in df.iterrows():
                direccion = row['direccion_completa']
                lat, lon = geocodificar_direccion(direccion)
                
                if lat and lon:
                    df.at[idx, 'latitud'] = lat
                    df.at[idx, 'longitud'] = lon
                    geocodificadas += 1
                else:
                    fallidas += 1
                
                # Mostrar progreso cada 10 propiedades
                if (geocodificadas + fallidas) % 10 == 0:
                    print(f"  Procesadas: {geocodificadas + fallidas}/{total} | Exitosas: {geocodificadas} | Fallidas: {fallidas}")
                
                # Delay de 1 segundo entre requests (requisito de Nominatim)
                time.sleep(1)
            
            print(f"\n✅ Geocodificación completada:")
            print(f"   - Exitosas: {geocodificadas} ({geocodificadas/total*100:.1f}%)")
            print(f"   - Fallidas: {fallidas} ({fallidas/total*100:.1f}%)")
            
            # Actualizar el dataframe
            dataframes[nombre_hoja] = df
            
            # Crear mapa si hay coordenadas válidas
            df_con_coordenadas = df[df['latitud'].notna() & df['longitud'].notna()]
            
            if len(df_con_coordenadas) > 0:
                print(f"\n🗺️ Creando mapa interactivo...")
                
                # Calcular centro del mapa (promedio de coordenadas)
                centro_lat = df_con_coordenadas['latitud'].mean()
                centro_lon = df_con_coordenadas['longitud'].mean()
                
                # Crear mapa base
                mapa = folium.Map(
                    location=[centro_lat, centro_lon],
                    zoom_start=12,
                    tiles='OpenStreetMap'
                )
                
                # Agregar clustering para mejor rendimiento
                marker_cluster = MarkerCluster().add_to(mapa)
                
                # Agregar marcadores para cada propiedad
                for idx, row in df_con_coordenadas.iterrows():
                    # Crear popup con información
                    popup_text = f"""
                    <b>{row.get('Dirección', 'N/A')}</b><br>
                    <b>Ubicación:</b> {row.get('Ubicación', 'N/A')}<br>
                    <b>Precio:</b> {row.get('Precio', 'N/A')}<br>
                    <b>Tipo:</b> {row.get('Tipo de propiedad', 'N/A')}<br>
                    <b>Estado:</b> {row.get('Estado', 'N/A')}<br>
                    <b>Operación:</b> {row.get('Tipo de operación', 'N/A')}
                    """
                    
                    # Color del marcador según tipo de operación
                    if row.get('Tipo de operación') == 'Venta':
                        color = 'red'
                    elif row.get('Tipo de operación') == 'Alquiler':
                        color = 'blue'
                    else:
                        color = 'gray'
                    
                    folium.Marker(
                        location=[row['latitud'], row['longitud']],
                        popup=folium.Popup(popup_text, max_width=300),
                        tooltip=row.get('Dirección', 'Propiedad'),
                        icon=folium.Icon(color=color, icon='home', prefix='fa')
                    ).add_to(marker_cluster)
                
                # Guardar mapa
                nombre_archivo = f'../data/portfolio/mapa_{nombre_hoja}.html'
                mapa.save(nombre_archivo)
                print(f"✅ Mapa guardado en: {nombre_archivo}")
                
                # Mostrar mapa en el notebook
                display(mapa)
                
                # Mostrar resumen de propiedades en el mapa
                print(f"\n📊 Resumen del mapa:")
                print(f"   - Propiedades geocodificadas: {len(df_con_coordenadas)}")
                print(f"   - Ventas (rojo): {len(df_con_coordenadas[df_con_coordenadas.get('Tipo de operación') == 'Venta'])}")
                print(f"   - Alquileres (azul): {len(df_con_coordenadas[df_con_coordenadas.get('Tipo de operación') == 'Alquiler'])}")
            else:
                print(f"⚠️ No hay coordenadas válidas para crear el mapa")
        else:
            print(f"⚠️ {nombre_hoja}: No se encontraron las columnas 'Dirección' y 'Ubicación'")
else:
    print("⚠️ No hay datos cargados para geocodificar")


In [88]:
# Análisis y KPIs del Portfolio
if len(dataframes) > 0:
    for nombre_hoja, df in dataframes.items():
        print(f"\n{'='*80}")
        print(f"📊 ANÁLISIS Y KPIs - {nombre_hoja.upper()}")
        print(f"{'='*80}\n")
        
        # KPI 1: Total de propiedades
        total_propiedades = len(df)
        print(f"📈 KPI 1: Total de Propiedades")
        print(f"   Total: {total_propiedades}")
        
        # KPI 2: Distribución por Tipo de Operación
        if 'Tipo de operación' in df.columns:
            print(f"\n📈 KPI 2: Distribución por Tipo de Operación")
            distribucion_operacion = df['Tipo de operación'].value_counts()
            for tipo, cantidad in distribucion_operacion.items():
                porcentaje = (cantidad / total_propiedades) * 100
                print(f"   {tipo}: {cantidad} ({porcentaje:.1f}%)")
        
        # KPI 3: Distribución por Tipo de Propiedad
        if 'Tipo de propiedad' in df.columns:
            print(f"\n📈 KPI 3: Distribución por Tipo de Propiedad")
            distribucion_tipo = df['Tipo de propiedad'].value_counts()
            for tipo, cantidad in distribucion_tipo.items():
                porcentaje = (cantidad / total_propiedades) * 100
                print(f"   {tipo}: {cantidad} ({porcentaje:.1f}%)")
        
        # KPI 4: Distribución por Estado
        if 'Estado' in df.columns:
            print(f"\n📈 KPI 4: Distribución por Estado")
            distribucion_estado = df['Estado'].value_counts()
            for estado, cantidad in distribucion_estado.items():
                porcentaje = (cantidad / total_propiedades) * 100
                print(f"   {estado}: {cantidad} ({porcentaje:.1f}%)")
        
        # KPI 5: Precios Promedio por Tipo de Operación
        if 'Precio_numerico' in df.columns and 'Tipo de operación' in df.columns:
            print(f"\n📈 KPI 5: Precios Promedio por Tipo de Operación")
            print(f"   ⚠️ Nota: Alquileres en pesos (ARS), Ventas en dólares (USD)")
            precios_por_tipo = df.groupby('Tipo de operación')['Precio_numerico'].agg(['mean', 'median', 'min', 'max', 'count'])
            precios_por_tipo = precios_por_tipo.rename(columns={
                'mean': 'Promedio',
                'median': 'Mediana',
                'min': 'Mínimo',
                'max': 'Máximo',
                'count': 'Cantidad'
            })
            # Agregar columna de moneda
            precios_por_tipo['Moneda'] = precios_por_tipo.index.map(lambda x: 'ARS' if x == 'Alquiler' else 'USD')
            display(precios_por_tipo)
        
        # KPI 6: Precios Promedio por Tipo de Propiedad
        if 'Precio_numerico' in df.columns and 'Tipo de propiedad' in df.columns:
            print(f"\n📈 KPI 6: Precios Promedio por Tipo de Propiedad")
            print(f"   ⚠️ Nota: Alquileres en pesos (ARS), Ventas en dólares (USD)")
            precios_por_tipo_prop = df.groupby('Tipo de propiedad')['Precio_numerico'].agg(['mean', 'median', 'min', 'max', 'count'])
            precios_por_tipo_prop = precios_por_tipo_prop.rename(columns={
                'mean': 'Promedio',
                'median': 'Mediana',
                'min': 'Mínimo',
                'max': 'Máximo',
                'count': 'Cantidad'
            })
            display(precios_por_tipo_prop)
        
        # KPI 7: Distribución por Ubicación (Top 10)
        if 'Ubicación' in df.columns:
            print(f"\n📈 KPI 7: Top 10 Ubicaciones")
            top_ubicaciones = df['Ubicación'].value_counts().head(10)
            for ubicacion, cantidad in top_ubicaciones.items():
                porcentaje = (cantidad / total_propiedades) * 100
                print(f"   {ubicacion}: {cantidad} ({porcentaje:.1f}%)")
        
        # KPI 8: Distribución por Usuario Asignado
        if 'Usuario asignado' in df.columns:
            print(f"\n📈 KPI 8: Distribución por Usuario Asignado")
            distribucion_usuario = df['Usuario asignado'].value_counts()
            for usuario, cantidad in distribucion_usuario.items():
                porcentaje = (cantidad / total_propiedades) * 100
                print(f"   {usuario}: {cantidad} ({porcentaje:.1f}%)")
        
        # KPI 9: Características de las Propiedades
        # Excluir TERRENO, CAMPO, GALPON, LOCAL del análisis de dormitorios y baños
        tipos_a_excluir = ['TERRENO', 'CAMPO', 'GALPON', 'LOCAL']
        
        if 'Dormitorios' in df.columns and 'Tipo de propiedad' in df.columns:
            print(f"\n📈 KPI 9: Distribución de Dormitorios")
            # Filtrar propiedades excluyendo los tipos especificados (manejar NaN)
            mask_dorm = df['Tipo de propiedad'].notna()
            mask_dorm = mask_dorm & (~df['Tipo de propiedad'].str.upper().isin(tipos_a_excluir))
            df_dorm = df[mask_dorm]
            total_prop_dorm = len(df_dorm)
            if total_prop_dorm > 0:
                # Convertir a numérico y eliminar NaN para ordenar correctamente
                dorm_clean = pd.to_numeric(df_dorm['Dormitorios'], errors='coerce')
                distribucion_dorm = dorm_clean.value_counts().sort_index()
                for dorm, cantidad in distribucion_dorm.items():
                    if pd.notna(dorm):
                        porcentaje = (cantidad / total_prop_dorm) * 100
                        print(f"   {int(dorm)} dormitorio(s): {cantidad} ({porcentaje:.1f}%)")
            else:
                print(f"   No hay propiedades válidas para analizar")
        
        if 'Baños' in df.columns and 'Tipo de propiedad' in df.columns:
            print(f"\n📈 KPI 10: Distribución de Baños")
            # Filtrar propiedades excluyendo los tipos especificados (manejar NaN)
            mask_banos = df['Tipo de propiedad'].notna()
            mask_banos = mask_banos & (~df['Tipo de propiedad'].str.upper().isin(tipos_a_excluir))
            df_banos = df[mask_banos]
            total_prop_banos = len(df_banos)
            if total_prop_banos > 0:
                # Convertir a numérico y eliminar NaN para ordenar correctamente
                banos_clean = pd.to_numeric(df_banos['Baños'], errors='coerce')
                distribucion_banos = banos_clean.value_counts().sort_index()
                for banos, cantidad in distribucion_banos.items():
                    if pd.notna(banos):
                        porcentaje = (cantidad / total_prop_banos) * 100
                        print(f"   {int(banos)} baño(s): {cantidad} ({porcentaje:.1f}%)")
            else:
                print(f"   No hay propiedades válidas para analizar")
        
        # KPI 11: Precio por m² de Ventas por Tipo de Propiedad
        if 'Precio_numerico' in df.columns and 'Tipo de operación' in df.columns and 'Superficie total' in df.columns and 'Tipo de propiedad' in df.columns and 'Estado' in df.columns:
            print(f"\n📈 KPI 11: Precio por m² de Ventas por Tipo de Propiedad (USD/m²)")
            print(f"   ⚠️ Solo se consideran propiedades con Estado 'Vigente'")
            
            # Filtrar solo ventas con Estado Vigente
            df_ventas = df[(df['Tipo de operación'] == 'Venta') & (df['Estado'] == 'Vigente')].copy()
            
            if len(df_ventas) > 0:
                # Excluir precio 0 o "consultar precio" (ya están como None en Precio_numerico)
                df_ventas = df_ventas[df_ventas['Precio_numerico'].notna() & (df_ventas['Precio_numerico'] > 0)]
                
                # Convertir Superficie total a numérico
                df_ventas['Superficie_total_num'] = pd.to_numeric(df_ventas['Superficie total'], errors='coerce')
                
                # Excluir cuando Superficie total no tiene datos o es 0
                df_ventas = df_ventas[df_ventas['Superficie_total_num'].notna() & (df_ventas['Superficie_total_num'] > 0)]
                
                if len(df_ventas) > 0:
                    # Calcular precio por m²
                    df_ventas['Precio_m2'] = df_ventas['Precio_numerico'] / df_ventas['Superficie_total_num']
                    
                    # Filtrar valores irrazonables usando IQR (Interquartile Range)
                    # Eliminar outliers que estén fuera de Q1 - 1.5*IQR y Q3 + 1.5*IQR
                    precio_m2_por_tipo = []
                    
                    for tipo_prop in df_ventas['Tipo de propiedad'].unique():
                        if pd.notna(tipo_prop):
                            df_tipo = df_ventas[df_ventas['Tipo de propiedad'] == tipo_prop].copy()
                            
                            if len(df_tipo) > 0:
                                Q1 = df_tipo['Precio_m2'].quantile(0.25)
                                Q3 = df_tipo['Precio_m2'].quantile(0.75)
                                IQR = Q3 - Q1
                                
                                # Filtrar outliers
                                lower_bound = Q1 - 1.5 * IQR
                                upper_bound = Q3 + 1.5 * IQR
                                
                                df_tipo_clean = df_tipo[
                                    (df_tipo['Precio_m2'] >= lower_bound) & 
                                    (df_tipo['Precio_m2'] <= upper_bound)
                                ]
                                
                                if len(df_tipo_clean) > 0:
                                    precio_m2_por_tipo.append({
                                        'Tipo de Propiedad': tipo_prop,
                                        'Promedio (USD/m²)': df_tipo_clean['Precio_m2'].mean(),
                                        'Mediana (USD/m²)': df_tipo_clean['Precio_m2'].median(),
                                        'Mínimo (USD/m²)': df_tipo_clean['Precio_m2'].min(),
                                        'Máximo (USD/m²)': df_tipo_clean['Precio_m2'].max(),
                                        'Cantidad': len(df_tipo_clean),
                                        'Valores excluidos (outliers)': len(df_tipo) - len(df_tipo_clean)
                                    })
                    
                    if precio_m2_por_tipo:
                        df_precio_m2 = pd.DataFrame(precio_m2_por_tipo)
                        df_precio_m2 = df_precio_m2.sort_values('Promedio (USD/m²)', ascending=False)
                        display(df_precio_m2)
                        
                        # Resumen general
                        total_ventas_iniciales = len(df[df['Tipo de operación'] == 'Venta'])
                        total_ventas_vigentes = len(df[(df['Tipo de operación'] == 'Venta') & (df['Estado'] == 'Vigente')])
                        total_ventas_validas = len(df_ventas)
                        total_excluidas = total_ventas_vigentes - total_ventas_validas
                        total_outliers = sum([item['Valores excluidos (outliers)'] for item in precio_m2_por_tipo])
                        print(f"\n   Resumen:")
                        print(f"   - Total ventas: {total_ventas_iniciales}")
                        print(f"   - Ventas con Estado 'Vigente': {total_ventas_vigentes}")
                        print(f"   - Ventas vigentes con precio y superficie válida: {total_ventas_validas}")
                        print(f"   - Ventas excluidas (precio 0/consultar precio o sin superficie): {total_excluidas}")
                        print(f"   - Outliers eliminados del análisis: {total_outliers}")
                    else:
                        print(f"   No hay datos suficientes para calcular precio por m²")
                else:
                    print(f"   No hay ventas vigentes con superficie total válida")
            else:
                print(f"   No hay ventas vigentes con precio válido para analizar")
        
        # KPI 12: Relación Tipo de Operación vs Tipo de Propiedad
        if 'Tipo de operación' in df.columns and 'Tipo de propiedad' in df.columns:
            print(f"\n📈 KPI 12: Relación Tipo de Operación vs Tipo de Propiedad")
            relacion_operacion_propiedad = pd.crosstab(df['Tipo de operación'], df['Tipo de propiedad'], margins=True)
            display(relacion_operacion_propiedad)
        
        # KPI 13: Relación Tipo de Operación vs Estado
        if 'Tipo de operación' in df.columns and 'Estado' in df.columns:
            print(f"\n📈 KPI 13: Relación Tipo de Operación vs Estado")
            relacion_operacion_estado = pd.crosstab(df['Tipo de operación'], df['Estado'], margins=True)
            display(relacion_operacion_estado)
        
        print(f"\n{'='*80}\n")
else:
    print("⚠️ No hay datos cargados para analizar")



📊 ANÁLISIS Y KPIs - ADINCO_EXPORT_20251201122116

📈 KPI 1: Total de Propiedades
   Total: 200

📈 KPI 2: Distribución por Tipo de Operación
   Venta: 144 (72.0%)
   Alquiler: 56 (28.0%)

📈 KPI 3: Distribución por Tipo de Propiedad
   Departamento: 88 (44.0%)
   Casa: 41 (20.5%)
   Terreno: 37 (18.5%)
   Galpón: 13 (6.5%)
   Quinta: 9 (4.5%)
   Local: 6 (3.0%)
   Negocio Especial: 2 (1.0%)
   Oficina: 1 (0.5%)
   Campo: 1 (0.5%)
   Hotel: 1 (0.5%)
   Edificio: 1 (0.5%)

📈 KPI 4: Distribución por Estado
   Vigente: 107 (53.5%)
   Alquilado: 36 (18.0%)
   Alquilado Incompleta: 32 (16.0%)
   Vendido: 13 (6.5%)
   Suspendido: 7 (3.5%)
   Vigente Incompleta: 4 (2.0%)
   Reservado: 1 (0.5%)

📈 KPI 5: Precios Promedio por Tipo de Operación
   ⚠️ Nota: Alquileres en pesos (ARS), Ventas en dólares (USD)


,Promedio,Mediana,Mínimo,Máximo,Cantidad,Moneda
Tipo de operación,,,,,,
Alquiler,676964.285714,572500.0,300000.0,2500000.0,56,ARS
Venta,101538.839286,65000.0,0.0,650000.0,112,USD



📈 KPI 6: Precios Promedio por Tipo de Propiedad
   ⚠️ Nota: Alquileres en pesos (ARS), Ventas en dólares (USD)


,Promedio,Mediana,Mínimo,Máximo,Cantidad
Tipo de propiedad,,,,,
Campo,260000.000000,260000.0,260000.0,260000.0,1
Casa,305727.500000,150000.0,0.0,2500000.0,40
Departamento,360579.264706,350000.0,0.0,1100000.0,68
Edificio,NaN,NaN,NaN,NaN,0
Galpón,570000.000000,190000.0,0.0,2200000.0,11
Hotel,NaN,NaN,NaN,NaN,0
Local,354000.000000,70000.0,0.0,1100000.0,5
Negocio Especial,157000.000000,157000.0,14000.0,300000.0,2
Oficina,500000.000000,500000.0,500000.0,500000.0,1



📈 KPI 7: Top 10 Ubicaciones
   Corrientes, Corrientes, Corrientes: 179 (89.5%)
   Paso de la Patria, Corrientes: 7 (3.5%)
   Riachuelo, Corrientes: 6 (3.0%)
   Santa Ana de los Guácaras, Corrientes: 5 (2.5%)
   Machagai, Chaco: 1 (0.5%)
   San Cosme, Corrientes: 1 (0.5%)
   San Luis del Palmar, Corrientes: 1 (0.5%)

📈 KPI 8: Distribución por Usuario Asignado
   Joaquin Francisco Garcia: 131 (65.5%)
   Federico Trim: 53 (26.5%)
   Alexis Burckwardt: 7 (3.5%)
   Juan Ignacio Esquercia: 6 (3.0%)
   Sofia Garcia: 2 (1.0%)
   Lautaro Gorostegui: 1 (0.5%)

📈 KPI 9: Distribución de Dormitorios
   1 dormitorio(s): 36 (23.1%)
   2 dormitorio(s): 39 (25.0%)
   3 dormitorio(s): 31 (19.9%)
   4 dormitorio(s): 11 (7.1%)
   5 dormitorio(s): 4 (2.6%)
   7 dormitorio(s): 1 (0.6%)
   20 dormitorio(s): 1 (0.6%)

📈 KPI 10: Distribución de Baños
   1 baño(s): 58 (37.2%)
   2 baño(s): 40 (25.6%)
   3 baño(s): 8 (5.1%)
   4 baño(s): 4 (2.6%)
   5 baño(s): 2 (1.3%)
   20 baño(s): 1 (0.6%)

📈 KPI 11: Precio 

Tipo de propiedad,Campo,Casa,Departamento,Edificio,Galpón,Hotel,Local,Negocio Especial,Oficina,Quinta,Terreno,All
Tipo de operación,,,,,,,,,,,,
Alquiler,0,6,40,0,5,0,2,1,1,1,0,56
Venta,1,35,48,1,8,1,4,1,0,8,37,144
All,1,41,88,1,13,1,6,2,1,9,37,200



📈 KPI 13: Relación Tipo de Operación vs Estado


Estado,Alquilado,Alquilado Incompleta,Reservado,Suspendido,Vendido,Vigente,Vigente Incompleta,All
Tipo de operación,,,,,,,,
Alquiler,34,0,0,1,0,21,0,56
Venta,2,32,1,6,13,86,4,144
All,36,32,1,7,13,107,4,200


## 6. Estructura Inicial de las Hojas


In [89]:
if len(dataframes) > 0:
    # Crear tabla resumen de estructura inicial
    resumen_estructura = []

    for nombre_hoja, df in dataframes.items():
        resumen_estructura.append({
            'Hoja': nombre_hoja,
            'Filas': len(df),
            'Columnas': len(df.columns),
            'Columnas Numéricas': len(df.select_dtypes(include=[np.number]).columns),
            'Columnas Texto': len(df.select_dtypes(include=['object']).columns),
            'Columnas Fecha': len(df.select_dtypes(include=['datetime64']).columns),
            'Valores Faltantes': df.isnull().sum().sum(),
            'Filas Duplicadas': df.duplicated().sum()
        })

    df_resumen_estructura = pd.DataFrame(resumen_estructura)
    print("📊 RESUMEN EJECUTIVO: Estructura Inicial")
    print("=" * 80)
    display(df_resumen_estructura)
else:
    print("⚠️ No hay datos cargados para analizar")


📊 RESUMEN EJECUTIVO: Estructura Inicial


,Hoja,Filas,Columnas,Columnas Numéricas,Columnas Texto,Columnas Fecha,Valores Faltantes,Filas Duplicadas
0,adinco_export_20251201122116,200,16,2,13,1,132,0


## 7. Detalle de Columnas por Hoja


In [90]:
if len(dataframes) > 0:
    # Tabla detallada de columnas por hoja
    detalle_columnas = []

    for nombre_hoja, df in dataframes.items():
        for col in df.columns:
            detalle_columnas.append({
                'Hoja': nombre_hoja,
                'Columna': col,
                'Tipo': str(df[col].dtype),
                'Valores Faltantes': df[col].isnull().sum(),
                'Porcentaje Faltantes': round((df[col].isnull().sum() / len(df)) * 100, 2) if len(df) > 0 else 0,
                'Valores Únicos': df[col].nunique()
            })

    df_detalle_columnas = pd.DataFrame(detalle_columnas)
    print("📋 DETALLE DE COLUMNAS POR HOJA")
    print("=" * 80)
    display(df_detalle_columnas)
else:
    print("⚠️ No hay datos cargados para analizar")


📋 DETALLE DE COLUMNAS POR HOJA


,Hoja,Columna,Tipo,Valores Faltantes,Porcentaje Faltantes,Valores Únicos
0,adinco_export_20251201122116,Creado,datetime64[ns],0,0.0,199
1,adinco_export_20251201122116,Slug,object,0,0.0,200
2,adinco_export_20251201122116,Estado,object,0,0.0,7
3,adinco_export_20251201122116,Dirección,object,0,0.0,176
4,adinco_export_20251201122116,Ubicación,object,0,0.0,7
5,adinco_export_20251201122116,Usuario asignado,object,0,0.0,6
6,adinco_export_20251201122116,Tipo de propiedad,object,0,0.0,11
7,adinco_export_20251201122116,Superficie total,object,0,0.0,81
8,adinco_export_20251201122116,Dormitorios,object,0,0.0,9
9,adinco_export_20251201122116,Baños,object,0,0.0,7


## 8. Vista Previa de Datos por Hoja


In [91]:
if len(dataframes) > 0:
    # Mostrar primeras filas de cada hoja para entender la estructura
    for nombre_hoja, df in dataframes.items():
        print(f"\n{'='*80}")
        print(f"📋 HOJA: {nombre_hoja.upper()}")
        print(f"{'='*80}")
        print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
        print(f"\nNombres de columnas:")
        for i, col in enumerate(df.columns, 1):
            print(f"  {i}. {col}")
        print(f"\nPrimeras 5 filas:")
        display(df.head())
        print("\n")
else:
    print("⚠️ No hay datos cargados para analizar")



📋 HOJA: ADINCO_EXPORT_20251201122116
Dimensiones: 200 filas × 16 columnas

Nombres de columnas:
  1. Creado
  2. Slug
  3. Estado
  4. Dirección
  5. Ubicación
  6. Usuario asignado
  7. Tipo de propiedad
  8. Superficie total
  9. Dormitorios
  10. Baños
  11. Expensas
  12. Precio
  13. Argenprop
  14. Superficie_total_num
  15. Precio_numerico
  16. Tipo de operación

Primeras 5 filas:


,Creado,Slug,Estado,Dirección,Ubicación,Usuario asignado,Tipo de propiedad,Superficie total,Dormitorios,Baños,Expensas,Precio,Argenprop,Superficie_total_num,Precio_numerico,Tipo de operación
0,2025-08-06 14:36:00,garc1-104,Vigente,Lomas Santa Ana,"Santa Ana de los Guácaras, Corrientes",Federico Trim,Quinta,-,3,1,48000,650000,-,NaN,650000.0,Alquiler
1,2025-09-09 15:24:00,garc1-185,Vigente,San Luis 590 8° 1,"Corrientes, Corrientes, Corrientes",Joaquin Francisco Garcia,Departamento,171 m2,3,4,480000,U$D 350.000,105 pts.,171.0,350000.0,Venta
2,2025-08-11 11:11:00,garc1-112,Vigente,Junin 2343,"Corrientes, Corrientes, Corrientes",Joaquin Francisco Garcia,Casa,275 m2,2,-,-,U$D 200.000,30 pts.,275.0,200000.0,Venta
3,2025-05-27 21:18:00,garc1-89,Vigente,Sicilia 4951,"Corrientes, Corrientes, Corrientes",Alexis Burckwardt,Casa,-,3,1,-,U$D 130.000,-,NaN,130000.0,Venta
4,2025-08-05 16:16:00,garc1-100,Vigente,Barrio Nuevo - Peatonal Cirilo Blanco,"Corrientes, Corrientes, Corrientes",Joaquin Francisco Garcia,Casa,248 m2,3,-,-,U$D 50.000,-,248.0,50000.0,Venta
